# XAS / CVS with CasidaPy

Import from the modular facade::

```python
from casidapy import xas
# or: from casidapy.xas import run_xas_gto, core_from_mf, ...
```

## Three workflows

| # | Name | One-shot | Host virtuals |
|---|------|----------|---------------|
| 1 | **GTO XAS** | `xas.run_xas_gto(mf, ...)` | Same molecule’s virtual MOs |
| 2 | **Manual inject** | `core_from_mf` → `inject_core_orbitals` → `run_cvs_tda` | Any GTO or PW kernel |
| 3 | **QE reconstruct** | `xas.run_xas_reconstruct(driver, ...)` or `CasidaKS_MPI.xas(...)` | QE conduction bands |

This notebook walks through (1) and (2) on water **O K-edge** (all-electron GTO).
Section (3) reconstructs the O 1s in a frozen QE embedding with **Hirshfeld partitioning** (`embed_mode="hirshfeld"`, the production default) and needs QEpy + O/H UPFs.

**Disclaimer.** STO-3G / small active spaces are for API teaching; production XAS needs a triple-ζ AE basis and many virtuals.

### Amarel / `libffi.so.7`

```bash
source /projectsn/mp1009_1/am4655/casidapy/tutorials/setup_env.sh
```

Then **Restart Kernel** (env must be set before the Python process starts).


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

from pyscf import gto, dft

_cwd = Path.cwd().resolve()
TUTORIALS = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "setup_env.sh").is_file() and (_cand / "paths.py").is_file():
        TUTORIALS = _cand
        break
if TUTORIALS is None:
    TUTORIALS = Path("/projectsn/mp1009_1/am4655/casidapy/tutorials")
if str(TUTORIALS) not in sys.path:
    sys.path.insert(0, str(TUTORIALS))
from paths import O_UPF, H_UPF  # noqa: E402
HERE = TUTORIALS
NB_DIR = _cwd

from casidapy import xas
from casidapy import extract_gto_kernel, CasidaOptions

print("xas helpers:", [n for n in dir(xas) if not n.startswith("_")][:12], "...")


## Shared AE water SCF

Oxygen is atom 0 → O K-edge.

In [ ]:
mol = gto.M(
    atom="""
    O  0.000000  0.000000  0.117300
    H  0.000000  0.757200 -0.469200
    H  0.000000 -0.757200 -0.469200
    """,
    basis="sto-3g",
    verbose=0,
)
mf = dft.RKS(mol).density_fit()
mf.xc = "pbe"
mf.kernel()
print(f"E = {mf.e_tot:.6f} Ha")
EDGE_O = 0  # oxygen atom index

## Workflow 1 — one-shot GTO XAS (`run_xas_gto`)

Under the hood: `core_from_mf` → build host kernel → `inject_core_orbitals` → CVS-TDA.

In [ ]:
res1, core1, kernel1 = xas.run_xas_gto(
    mf,
    edge="K",
    edge_atom_indices=[EDGE_O],
    n_unocc=12,
    n_states=10,
    use_df=True,
    verbose=False,
)
print(xas.summarize_xas(res1, core1))
print("kernel n_occ (cores only) =", kernel1.n_occ, " n_virt =", kernel1.n_unocc)

## Workflow 2 — manual inject (same physics, explicit steps)

Use this when you already have a PW or GTO host, or want to inspect cores first.

In [ ]:
# 2a. Pull O 1s from the AE mf
core2 = xas.core_from_mf(mf, edge="K", edge_atom_indices=[EDGE_O])
print("core energies (Ha):", np.round(core2.energies, 4))
print("shell / edge:", core2.shell, core2.edge)

# 2b. Host with real molecular virtuals (placeholder occupied slot)
host, opts = extract_gto_kernel(
    mf, n_occ=1, n_unocc=12, n_states=10, use_df=True, k_cache_max=0
)

# 2c. Replace occupied active space with the core MO(s)
xas.inject_core_orbitals(host, core2.energies, core2.mo_coeff)

# 2d. CVS-TDA
res2 = xas.run_cvs_tda(host, opts, n_states=10)
print(xas.summarize_xas(res2, core2))

print("Δω₀ (manual − one-shot) =",
      float(xas.omega_ev(res2)[0] - xas.omega_ev(res1)[0]), "eV")

## Plot Workflow 1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
xas.plot_sticks(
    res1, ax=ax, label="O K-edge (GTO CVS)", color="C0", broaden_sigma_ev=0.4
)
ax.set_title("H₂O O K-edge — STO-3G / PBE (tutorial quality)")
plt.tight_layout()
plt.show()

## Workflow 3 — reconstruct AE core in QE embedding (Hirshfeld)

Needs QEpy + ONCV UPFs in `tutorials/` (resolved via `paths.py`).

Physics: PP SCF → Hirshfeld `V_env` → one-atom AE SCF for frozen shells → inject into PW virtuals:

$$
w_A=\frac{\tilde\rho_A}{\sum_B\tilde\rho_B},\quad
\rho_\mathrm{env}=(1-w_A)\,\rho_\mathrm{tot},\quad
V_\mathrm{env}=V_\mathrm{ionic}+v_H[\rho_\mathrm{env}]+v_\mathrm{xc}[\rho_\mathrm{env}]
$$

with $V_\mathrm{ionic}=v_\mathrm{ltot}-V_\mathrm{loc}^\mathrm{QE}(A)$ (`vloc_source="qepy"`).

**Defaults / fair comparison**

- `embed_mode="hirshfeld"` (production default of `CasidaKS_MPI.xas` / `run_xas_reconstruct`)
- `eps_core_reference_ha` — rigid-shift the reconstructed ε_core onto the full-AE GTO core from Workflow 1 (removes absolute embedding offset so spectra can be compared)
- When overlaying sticks, `normalize=True` scales each spectrum to `∑f = 1` (GTO vs PW hosts have different absolute oscillator-strength scales)

Legacy embed modes: `loc_only`, `damp_vhxc` / `scale_vhxc`. Partition detail: `hirshfeld_partition_demo.ipynb`.

```python
casida.xas(
    reconstruct=True, edge_atom=0, edge="K",
    embed_mode="hirshfeld",
    eps_core_reference_ha=float(np.min(core1.energies)),
    ...
)
```


In [ ]:
import os

RUN_RECONSTRUCT = O_UPF.is_file() and H_UPF.is_file()

if not RUN_RECONSTRUCT:
    print("Skipping Workflow 3 — UPFs not found in", HERE)
else:
    from qepy.driver import Driver
    from dftpy.functional.xc import XC
    from casidapy import CasidaKS_MPI

    # Align reconstruct ε_core to the full-AE GTO core from Workflow 1
    eps_ref = float(np.min(core1.energies))
    print(f"eps_core_reference_ha = {eps_ref:.6f} Ha  ({eps_ref * xas.HA_TO_EV:.2f} eV)")

    qe_dir = (NB_DIR / "_xas_recon_out").resolve()
    qe_dir.mkdir(exist_ok=True)
    qe_options = {
        "&control": {
            "calculation": "'scf'",
            "pseudo_dir": f"'{HERE.resolve()}/'",
            "outdir": f"'{qe_dir}/'",
            "prefix": "'h2o_xas'",
            "verbosity": "'low'",
        },
        "&system": {
            "ibrav": 0,
            "nat": 3,
            "ntyp": 2,
            "ecutwfc": 30.0,
            "ecutrho": 120.0,
            "nbnd": 20,
            "nosym": True,
            "occupations": "'fixed'",
        },
        "&electrons": {"conv_thr": 1e-6, "mixing_beta": 0.7},
        "atomic_species": [
            "O  15.9994  O_ONCV_PBE-1.2.upf",
            "H   1.0079  H_ONCV_PBE-1.2.upf",
        ],
        "atomic_positions angstrom": [
            "O  6.000000  6.000000  6.117300",
            "H  6.000000  6.757200  5.530800",
            "H  6.000000  5.242800  5.530800",
        ],
        "cell_parameters angstrom": [
            "12.0  0.0  0.0",
            "0.0  12.0  0.0",
            "0.0  0.0  12.0",
        ],
        "k_points gamma": [],
    }
    # QEpy reads input_tmp.in from CWD — isolate under qe_dir
    prev = os.getcwd()
    try:
        os.chdir(qe_dir)
        driver = Driver(qe_options=qe_options, logfile=str(qe_dir / "scf.log"))
        driver.scf()
    finally:
        os.chdir(prev)

    rho = driver.data2field(driver.get_density())
    casida = CasidaKS_MPI(rho, XC(xc="PBE"), driver=driver)
    res3, core3, kernel3, mf3 = casida.xas(
        reconstruct=True,
        edge_atom=0,
        edge="K",
        basis="sto-3g",
        xc="pbe",
        n_virt=8,
        n_states=6,
        upf_path=str(HERE / "O_ONCV_PBE-1.2.upf"),
        embed_mode="hirshfeld",
        vloc_source="qepy",
        eps_core_reference_ha=eps_ref,
        verbose=False,
    )
    print(xas.summarize_xas(res3, core3))
    meta = getattr(core3, "meta", {}).get("reconstruct_meta", {})
    print(
        "embed:", meta.get("embed_mode"),
        " vloc:", meta.get("vloc_source"),
        " gauge_shift:", meta.get("gauge_shift_ha"),
        " core_ref_shift:", meta.get("core_reference_shift_ha"),
    )
    print(
        "Δω₀ (recon − GTO) =",
        float(xas.omega_ev(res3)[0] - xas.omega_ev(res1)[0]),
        "eV",
    )

    # Functional form (equivalent):
    # res3, core3, kernel3, mf3 = xas.run_xas_reconstruct(
    #     driver, 0, edge="K", basis="sto-3g", xc="pbe",
    #     n_virt=8, n_states=6, embed_mode="hirshfeld", vloc_source="qepy",
    #     eps_core_reference_ha=eps_ref,
    # )

    driver.stop()

    # Overlay with ∑f = 1 so GTO vs PW intensities are comparable
    fig, ax = plt.subplots(figsize=(7, 3.5))
    xas.plot_sticks(
        res1, ax=ax, label="GTO CVS", color="C0",
        broaden_sigma_ev=0.5, normalize=True,
    )
    xas.plot_sticks(
        res3, ax=ax, label="QE Hirshfeld (+ε_core ref)", color="C2",
        broaden_sigma_ev=0.5, normalize=True,
    )
    ax.set_title("O K-edge — GTO vs Hirshfeld GTO+PW (ref-aligned, ∑f=1)")
    plt.tight_layout()
    plt.show()


## Where the code lives

```
casidapy.xas                 # facade (import this)
  ├─ cvs.py                  # CoreOrbitals, inject, run_cvs_gto_from_mf
  ├─ reconstruct.py          # V_env → AE core → PW CVS
  └─ spectrum.py             # plot_sticks, stick_spectrum, …
casidapy.embed               # AE embedding potentials
  ├─ potential.py            # V_ionic peel + embed_mode dispatch (default: hirshfeld)
  └─ hirshfeld.py            # w_A · ρ_tot → v_Hxc[ρ_env]
```
